# Notebook 1: Testing Model Implementations
This notebook tests all your model implementations to ensure they work correctly.

In [1]:
import os
import sys

# Change working directory to project root
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

# Add src to path if not already handled
if 'src' not in sys.path:
    sys.path.append('src')

print(f"Current working directory: {os.getcwd()}")

Current working directory: /Users/manue/Desktop/final-project-rdgzmanuel


In [2]:
import sys
sys.path.append('../src')

import torch
import numpy as np
import pickle
from pathlib import Path
import matplotlib.pyplot as plt

from data_preprocessing import load_splits, GTSRBDataset, get_transforms, GTSRBConfig
from models import LogisticRegressionHOG, ShallowCNN, PretrainedMobileNetV2, PretrainedResNet18, get_model
from torch.utils.data import DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cpu


## 1. Test Logistic Regression on HOG Features

In [3]:
# Load HOG features
processed_dir = Path('data/processed')

with open(processed_dir / 'train_hog_features.pkl', 'rb') as f:
    X_train_hog = pickle.load(f)

with open(processed_dir / 'val_hog_features.pkl', 'rb') as f:
    X_val_hog = pickle.load(f)

splits = load_splits(processed_dir)
_, y_train = splits['train']
_, y_val = splits['val']

print(f"HOG features shape: {X_train_hog.shape}")
print(f"Training samples: {len(y_train)}")
print(f"Validation samples: {len(y_val)}")

HOG features shape: (36287, 1764)
Training samples: 36287
Validation samples: 7776


In [4]:
# Test Logistic Regression
print("Testing Logistic Regression...")
lr_model = LogisticRegressionHOG(num_classes=43, max_iter=500)

# Train on small subset for quick test
X_train_subset = X_train_hog[:1000]
y_train_subset = y_train[:1000]

lr_model.train(X_train_subset, y_train_subset)
print("Training completed")

# Test predictions
y_pred = lr_model.predict(X_val_hog[:100])
y_proba = lr_model.predict_proba(X_val_hog[:100])

print(f"✓ Predictions shape: {y_pred.shape}")
print(f"✓ Probabilities shape: {y_proba.shape}")
print(f"✓ Sample predictions: {y_pred[:10]}")
print(f"✓ Probability sum (should be ~1.0): {y_proba[0].sum():.4f}")

from sklearn.metrics import accuracy_score
acc = accuracy_score(y_val[:100], y_pred)
print(f"\n✓ Accuracy on 100 validation samples: {acc:.4f}")
print("\nLogistic Regression implementation is working")

Testing Logistic Regression...
Training completed
✓ Predictions shape: (100,)
✓ Probabilities shape: (100, 43)
✓ Sample predictions: [14 24  3 12  2  7 24 16  2  5]
✓ Probability sum (should be ~1.0): 1.0000

✓ Accuracy on 100 validation samples: 0.8500

Logistic Regression implementation is working


## 2. Test Shallow CNN

In [5]:
# Create small dataset for testing
val_dataset = GTSRBDataset(
    splits['val'][0][:100],
    splits['val'][1][:100],
    transform=get_transforms(augment=False)
)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

print(f"Test dataset size: {len(val_dataset)}")

Test dataset size: 100


In [6]:
# Test Shallow CNN
print("Testing Shallow CNN...")
shallow_cnn = ShallowCNN(num_classes=43).to(device)
print(f"✓ Model created with {sum(p.numel() for p in shallow_cnn.parameters())} parameters")

# Test forward pass
test_input = torch.randn(2, 3, 64, 64).to(device)
output = shallow_cnn(test_input)
print(f"✓ Output shape: {output.shape} (expected: [2, 43])")

# Test with real data
shallow_cnn.eval()
with torch.no_grad():
    for images, _ in val_loader:
        images = images.to(device)
        outputs = shallow_cnn(images)
        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(outputs, dim=1)
        
        print(f"✓ Batch output shape: {outputs.shape}")
        print(f"✓ Predictions: {preds.cpu().numpy()[:5]}")
        print(f"✓ Max probability: {probs.max().item():.4f}")
        break

print("\nShallow CNN implementation is working")

Testing Shallow CNN...
✓ Model created with 34699 parameters
✓ Output shape: torch.Size([2, 43]) (expected: [2, 43])
✓ Batch output shape: torch.Size([8, 43])
✓ Predictions: [34  9 20 22 20]
✓ Max probability: 0.0447

Shallow CNN implementation is working


## 3. Test Pretrained MobileNetV2

In [7]:
# Test MobileNetV2
print("Testing MobileNetV2...")
mobilenet = PretrainedMobileNetV2(num_classes=43, pretrained=True).to(device)
print(f"✓ Model created with {sum(p.numel() for p in mobilenet.parameters())} parameters")

# Test forward pass
test_input = torch.randn(2, 3, 64, 64).to(device)
output = mobilenet(test_input)
print(f"✓ Output shape: {output.shape} (expected: [2, 43])")

# Test freezing
print("\nTesting freeze_backbone...")
mobilenet.freeze_backbone()
trainable_params = sum(p.numel() for p in mobilenet.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in mobilenet.parameters())
print(f"✓ Trainable parameters after freezing: {trainable_params}/{total_params}")
print(f"✓ Percentage trainable: {trainable_params/total_params*100:.2f}%")

# Test unfreezing
print("\nTesting unfreeze_last_blocks...")
mobilenet.unfreeze_last_blocks(num_blocks=2)
trainable_params = sum(p.numel() for p in mobilenet.parameters() if p.requires_grad)
print(f"✓ Trainable parameters after unfreezing: {trainable_params}/{total_params}")
print(f"✓ Percentage trainable: {trainable_params/total_params*100:.2f}%")

print("\nMobileNetV2 implementation is working")

Testing MobileNetV2...
✓ Model created with 2278955 parameters
✓ Output shape: torch.Size([2, 43]) (expected: [2, 43])

Testing freeze_backbone...
✓ Trainable parameters after freezing: 55083/2278955
✓ Percentage trainable: 2.42%

Testing unfreeze_last_blocks...
✓ Trainable parameters after unfreezing: 941163/2278955
✓ Percentage trainable: 41.30%

MobileNetV2 implementation is working


## 4. Test Pretrained ResNet18

In [8]:
# Test ResNet18
print("Testing ResNet18...")
resnet = PretrainedResNet18(num_classes=43, pretrained=True).to(device)
print(f"✓ Model created with {sum(p.numel() for p in resnet.parameters())} parameters")

# Test forward pass
test_input = torch.randn(2, 3, 64, 64).to(device)
output = resnet(test_input)
print(f"✓ Output shape: {output.shape} (expected: [2, 43])")

# Test freezing
print("\nTesting freeze_backbone...")
resnet.freeze_backbone()
trainable_params = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in resnet.parameters())
print(f"✓ Trainable parameters after freezing: {trainable_params}/{total_params}")
print(f"✓ Percentage trainable: {trainable_params/total_params*100:.2f}%")

# Test unfreezing
print("\nTesting unfreeze_last_block...")
resnet.unfreeze_last_block()
trainable_params = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
print(f"✓ Trainable parameters after unfreezing: {trainable_params}/{total_params}")
print(f"✓ Percentage trainable: {trainable_params/total_params*100:.2f}%")

print("\nResNet18 implementation is working")

Testing ResNet18...
✓ Model created with 11198571 parameters
✓ Output shape: torch.Size([2, 43]) (expected: [2, 43])

Testing freeze_backbone...
✓ Trainable parameters after freezing: 22059/11198571
✓ Percentage trainable: 0.20%

Testing unfreeze_last_block...
✓ Trainable parameters after unfreezing: 8415787/11198571
✓ Percentage trainable: 75.15%

ResNet18 implementation is working


## 5. Test Model Factory Function

In [9]:
print("Testing get_model factory function...")

for model_name in ['shallow_cnn', 'mobilenet', 'resnet18']:
    print(f"\nCreating {model_name}...")
    model = get_model(model_name, num_classes=43, pretrained=True, device=device)
    
    # Test forward pass
    test_input = torch.randn(2, 3, 64, 64).to(device)
    output = model(test_input)
    
    print(f"✓ {model_name} output shape: {output.shape}")
    print(f"✓ Parameters: {sum(p.numel() for p in model.parameters())}")

print("\nAll model implementations are working correctly")

Testing get_model factory function...

Creating shallow_cnn...
✓ shallow_cnn output shape: torch.Size([2, 43])
✓ Parameters: 34699

Creating mobilenet...
✓ mobilenet output shape: torch.Size([2, 43])
✓ Parameters: 2278955

Creating resnet18...
✓ resnet18 output shape: torch.Size([2, 43])
✓ Parameters: 11198571

All model implementations are working correctly
